In [1]:
import cv2
import numpy as np
import pandas as pd
import torch
from ultralytics import YOLO

In [2]:
# =========================
# CONFIG GPU
# =========================
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


In [3]:
# =========================
# MODELO
# =========================
model = YOLO("../Models/yolov8x-pose-p6.pt")
model.to(device)

torch.backends.cudnn.benchmark = True

In [4]:
# =========================
# VIDEO INPUT
# =========================
cap = cv2.VideoCapture("video.mp4")

frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

In [5]:
# =========================
# VIDEO OUTPUT (PREVIEW)
# =========================
out = cv2.VideoWriter(
    "preview_pose.mp4",
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (frame_width, frame_height)
)


In [ ]:
# =========================
# COLETA
# =========================
data = []
frame_id = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results = model.predict(frame, device=device, verbose=False)

    for r in results:
        if r.keypoints is None:
            continue

        kpts = r.keypoints.xy.cpu().numpy()

        for person in kpts:
            for joint_id, (x, y) in enumerate(person):

                # salvar dados
                data.append({
                    "frame": frame_id,
                    "joint": joint_id,
                    "x": float(x),
                    "y": float(y)
                })

                # desenhar no frame (preview)
                cv2.circle(
                    frame,
                    (int(x), int(y)),
                    3,
                    (0, 255, 0),
                    -1
                )

    # escreve frame no vídeo de saída
    out.write(frame)

    frame_id += 1

cap.release()
out.release()

df = pd.DataFrame(data)

print("Frames processados:", frame_id)
print("Vídeo salvo como preview_pose.mp4")